# GIFT-Eval Benchmark Analysis

Downloads all model results from the [GIFT-Eval HuggingFace leaderboard](https://huggingface.co/spaces/Salesforce/GIFT-Eval), builds a flat dataframe, and produces a series of diagnostic plots comparing statistical baselines against foundation models.

**Sections**
1. Setup & function definitions
2. Load raw data
3. Build analysis dataframe
4. Head-to-head (statistical vs. foundation)
5. Summary statistics
6. Plots 1–9


## Setup

Imports, constants, and all function definitions.


In [ ]:
%matplotlib inline

import io
import warnings

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

BASE_URL = "https://huggingface.co/spaces/Salesforce/GIFT-Eval/raw/main/results"

MODEL_DIRS = [
    "Chronos_small",
    "CleanTS-65M",
    "Credence",
    "DLinear",
    "DeOSAlphaTimeGPTPredictor-2025",
    "FFM",
    "FlowState-9.1M",
    "Kairos_10m",
    "Kairos_23m",
    "Kairos_50m",
    "Lag-Llama",
    "Lingjiang",
    "Migas-1.0",
    "Moirai2",
    "MoiraiAgent",
    "Moirai_base",
    "Moirai_large",
    "Moirai_small",
    "N-BEATS",
    "PatchTST-FM-r1",
    "PatchTST",
    "Reverso-Nano",
    "Reverso-Small",
    "Reverso",
    "Samay",
    "Synapse",
    "TSOrchestra",
    "TTM-R1-Pretrained",
    "TTM-R2-Finetuned",
    "TTM-R2-Pretrained",
    "TempoPFN",
    "TiRex",
    "TimeCopilot",
    "TimesFM-2.5",
    "Toto_Open_Base_1.0",
    "Xihe-max",
    "Xihe-ultra",
    "YingLong_110m",
    "YingLong_300m",
    "YingLong_50m",
    "YingLong_6m",
    "auto_arima",
    "auto_ets",
    "auto_theta",
    "chronos-2",
    "chronos-2-synth",
    "chronos_base",
    "chronos_bolt_base",
]

METRIC = "MASE[0.5]"

PAL = {
    "statistical": "#e69f00",
    "foundation": "#0072b2",
    "deep-learning": "#009e73",
    "other": "#cc79a7",
}

FREQ_PERIOD_HOURS = {
    "1T": 1 / 60,
    "5T": 5 / 60,
    "10T": 10 / 60,
    "15T": 15 / 60,
    "30T": 0.5,
    "H": 1,
    "4H": 4,
    "D": 24,
    "W": 168,
    "M": 720,
    "ME": 720,
    "Q": 2160,
    "QE": 2160,
    "A": 8760,
    "YE": 8760,
}

# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------


def fetch_config(model_dir: str) -> dict:
    url = f"{BASE_URL}/{model_dir}/config.json"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return {}


def fetch_results(model_dir: str) -> pd.DataFrame | None:
    url = f"{BASE_URL}/{model_dir}/all_results.csv"
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            return pd.read_csv(io.StringIO(r.text))
    except Exception:
        pass
    return None


def load_all() -> pd.DataFrame:
    frames = []
    skipped = []
    for model_dir in MODEL_DIRS:
        cfg = fetch_config(model_dir)
        df = fetch_results(model_dir)
        if df is None or df.empty:
            skipped.append(model_dir)
            continue
        for k, v in cfg.items():
            df[f"cfg_{k}"] = v
        df["model_dir"] = model_dir
        frames.append(df)

    if skipped:
        print(f"Skipped (no data): {skipped}")
    print(f"Loaded {len(frames)} models")
    return pd.concat(frames, ignore_index=True)


def build_dataframe(raw: pd.DataFrame) -> pd.DataFrame:
    raw = raw.copy()
    raw.columns = [c.replace("eval_metrics/", "") for c in raw.columns]

    parts = raw["dataset"].str.split("/", expand=True, n=2)
    raw["ds_name"] = parts[0]
    raw["ds_freq"] = parts[1]
    raw["ds_horizon"] = parts[2]

    raw["model_name"] = (
        raw["cfg_model"].fillna(raw["model_dir"])
        if "cfg_model" in raw.columns
        else raw["model_dir"]
    )

    raw["model_type"] = raw.get("cfg_model_type", pd.Series(dtype=str))

    def categorise(row):
        mt = str(row.get("model_type") or "").lower()
        if "statistical" in mt:
            return "statistical"
        if mt in ("pretrained", "zero-shot", "foundation"):
            return "foundation"
        if "deep" in mt or "learning" in mt:
            return "deep-learning"
        return "other"

    raw["category"] = raw.apply(categorise, axis=1)
    raw["org"] = raw.get("cfg_org", pd.Series(dtype=str))

    df = raw.dropna(subset=[METRIC]).copy()
    print(
        f"Rows: {len(df)}, Models: {df['model_name'].nunique()}, "
        f"Datasets: {df['dataset'].nunique()}"
    )
    return df


# ---------------------------------------------------------------------------
# Feature engineering helpers
# ---------------------------------------------------------------------------


def map_freq_hours(f: str) -> float:
    f = str(f).upper()
    for k, v in sorted(FREQ_PERIOD_HOURS.items(), key=lambda x: -len(x[0])):
        if f.endswith(k):
            prefix = f[: -len(k)]
            mult = int(prefix) if prefix.isdigit() else 1
            return v * mult
    return float("nan")


def freq_bucket(f: str) -> str:
    h = map_freq_hours(f)
    if np.isnan(h):
        return "unknown"
    if h < 1:
        return "sub-hourly"
    if h <= 4:
        return "hourly"
    if h <= 48:
        return "daily"
    return "weekly+"


# ---------------------------------------------------------------------------
# Analysis helpers
# ---------------------------------------------------------------------------


def build_head2head(df: pd.DataFrame) -> pd.DataFrame:
    other = df[df["category"] != "foundation"]
    fnd = df[df["category"] == "foundation"]

    other_best = other.groupby("dataset")[METRIC].min().rename("other_best")
    found_best = fnd.groupby("dataset")[METRIC].min().rename("found_best")
    found_med = fnd.groupby("dataset")[METRIC].median().rename("found_median")
    found_std = fnd.groupby("dataset")[METRIC].std().rename("found_std")

    ds_meta = (
        df[["dataset", "ds_name", "ds_freq", "ds_horizon", "domain", "num_variates"]]
        .drop_duplicates("dataset")
        .set_index("dataset")
    )

    h2h = pd.concat([other_best, found_best, found_med, found_std], axis=1).dropna(
        subset=["other_best", "found_best"]
    )
    h2h = h2h.join(ds_meta)

    h2h["log_ratio"] = np.log(h2h["found_best"] / h2h["other_best"])
    h2h["other_wins"] = h2h["log_ratio"] > 0
    h2h["winner"] = np.where(h2h["other_wins"], "other", "foundation")
    h2h["freq_bucket"] = h2h["ds_freq"].apply(freq_bucket)
    h2h["horizon"] = pd.Categorical(
        h2h["ds_horizon"], ["short", "medium", "long"], ordered=True
    )
    h2h["cv"] = h2h["found_std"] / h2h["found_median"]
    h2h["log_freq_hours"] = h2h["ds_freq"].apply(map_freq_hours).pipe(np.log1p)
    h2h["horizon_ord"] = h2h["horizon"].cat.codes
    h2h["log_variates"] = np.log1p(h2h["num_variates"].astype(float))
    return h2h


def build_rank_pivot(df: pd.DataFrame) -> pd.DataFrame:
    pivot = df.pivot_table(
        index="dataset", columns="model_name", values=METRIC, aggfunc="median"
    )
    keep = pivot[pivot.notna().sum(axis=1) >= pivot.shape[1] * 0.4]
    ranks = keep.rank(axis=1, method="average", na_option="keep")
    mean_rank = ranks.mean().sort_values()
    return ranks[mean_rank.index]


# ---------------------------------------------------------------------------
# Plots
# ---------------------------------------------------------------------------


def plot_overall_ranking(df: pd.DataFrame):
    model_avg = (
        df.groupby(["model_name", "category"])[METRIC]
        .median()
        .reset_index()
        .sort_values(METRIC)
    )
    colors = model_avg["category"].map(PAL)

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.bar(model_avg["model_name"], model_avg[METRIC], color=colors)
    ax.set_xticklabels(model_avg["model_name"], rotation=60, ha="right", fontsize=7)
    ax.set_ylabel("Median MASE across datasets")
    ax.set_title("Overall benchmark ranking (lower = better)")
    ax.legend(
        handles=[mpatches.Patch(color=v, label=k) for k, v in PAL.items()],
        loc="upper left",
    )
    plt.show()


def plot_mase_by_domain(df: pd.DataFrame):
    domain_cat = df.groupby(["domain", "category"])[METRIC].median().unstack("category")
    fig, ax = plt.subplots(figsize=(12, 4))
    domain_cat.plot(
        kind="bar",
        ax=ax,
        color=[PAL.get(c, "grey") for c in domain_cat.columns],
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_ylabel("Median MASE")
    ax.set_title("Median MASE by domain and model category")
    ax.legend(title="category")
    plt.show()


def plot_waterfall(h2h: pd.DataFrame):
    srt = h2h.sort_values("log_ratio", ascending=False)
    colors = ["#e69f00" if v > 0 else "#0072b2" for v in srt["log_ratio"]]

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(range(len(srt)), srt["log_ratio"], color=colors, width=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(srt)))
    ax.set_xticklabels(srt.index, rotation=90, fontsize=5)
    ax.set_ylabel(
        "log(found_best / other_best)\n(+) = other wins,  (−) = foundation wins"
    )
    ax.set_title("Per-dataset advantage: other (orange) vs foundation (blue)")
    ax.legend(
        handles=[
            mpatches.Patch(color="#e69f00", label="other wins"),
            mpatches.Patch(color="#0072b2", label="foundation wins"),
        ]
    )
    plt.show()


def plot_scatter_drivers(h2h: pd.DataFrame):
    domains = h2h["domain"].dropna().unique()
    domain_pal = dict(zip(domains, plt.cm.tab10.colors[: len(domains)]))

    plot_cfg = [
        ("log_freq_hours", "Log frequency period (hours)"),
        ("horizon_ord", "Horizon  (0=short, 1=med, 2=long)"),
        ("log_variates", "Log num variates"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for (xvar, xlabel), ax in zip(plot_cfg, axes):
        sub = h2h[[xvar, "log_ratio", "domain"]].dropna()
        for dom, grp in sub.groupby("domain"):
            ax.scatter(
                grp[xvar],
                grp["log_ratio"],
                label=dom,
                alpha=0.7,
                s=30,
                color=domain_pal.get(dom, "grey"),
            )
        m, b = np.polyfit(sub[xvar], sub["log_ratio"], 1)
        xs = np.linspace(sub[xvar].min(), sub[xvar].max(), 100)
        ax.plot(xs, m * xs + b, "k--", linewidth=1.5)
        ax.axhline(0, color="grey", linewidth=0.8)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("log(found/other)  +ve = other wins")
        r, p = stats.pearsonr(sub[xvar], sub["log_ratio"])
        ax.set_title(f"r={r:+.2f}  p={p:.3f}")

    axes[0].legend(fontsize=7, loc="upper right")
    fig.suptitle("Drivers of other vs foundation advantage", y=1.01)
    fig.tight_layout()
    plt.show()


def plot_boxplots(h2h: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    order_dom = (
        h2h.groupby("domain")["log_ratio"].median().sort_values(ascending=False).index
    )
    sns.boxplot(
        data=h2h,
        x="domain",
        y="log_ratio",
        order=order_dom,
        ax=axes[0],
        palette="tab10",
    )
    axes[0].axhline(0, color="black", linewidth=1)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha="right")
    axes[0].set_title("Other vs foundation advantage by domain")
    axes[0].set_ylabel("log(found/other)  +ve = other wins")

    sns.boxplot(
        data=h2h,
        x="ds_horizon",
        y="log_ratio",
        order=["short", "medium", "long"],
        ax=axes[1],
        palette="Set2",
    )
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Other vs foundation advantage by horizon")
    axes[1].set_ylabel("")

    fig.tight_layout()
    plt.show()


def plot_freq_stacked_bar(h2h: pd.DataFrame):
    ct = (
        pd.crosstab(h2h["freq_bucket"], h2h["winner"], normalize="index") * 100
    ).reindex(["sub-hourly", "hourly", "daily", "weekly+"])
    for col in ("other", "foundation"):
        if col not in ct.columns:
            ct[col] = 0.0

    fig, ax = plt.subplots(figsize=(7, 4))
    ct[["foundation", "other"]].plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=["#0072b2", "#e69f00"],
    )
    ax.set_ylabel("% of datasets")
    ax.set_title("Who wins by data frequency?")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title="winner", loc="lower right")
    fig.tight_layout()
    plt.show()


def plot_rank_heatmap(df: pd.DataFrame):
    ranks = build_rank_pivot(df)

    ds_meta = df[["dataset", "domain"]].drop_duplicates("dataset").set_index("dataset")
    domain_order = ds_meta.reindex(ranks.index)["domain"].sort_values()
    ranks = ranks.loc[domain_order.index]

    n_ds = len(ranks)
    fig, ax = plt.subplots(figsize=(22, max(8, n_ds * 0.22)))
    sns.heatmap(
        ranks.T,
        cmap="RdYlGn_r",
        ax=ax,
        linewidths=0,
        cbar_kws={"label": "Rank (1=best)"},
        xticklabels=True,
        yticklabels=True,
    )
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=5)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7)
    ax.set_title(
        "Per-dataset rank heatmap (models sorted by mean rank, datasets sorted by domain)",
        pad=10,
    )

    prev, x = None, 0
    for ds in domain_order.index:
        dom = domain_order[ds]
        if dom != prev and prev is not None:
            ax.axvline(x, color="white", linewidth=2)
        prev = dom
        x += 1

    fig.tight_layout()
    plt.show()


def plot_clustermap(df: pd.DataFrame):
    ranks = build_rank_pivot(df)
    norm = ranks.div(ranks.max(axis=1), axis=0)
    clean = norm.dropna(thresh=int(norm.shape[1] * 0.5), axis=0)
    clean = clean.dropna(thresh=int(clean.shape[0] * 0.5), axis=1)
    clean = clean.fillna(clean.median())

    g = sns.clustermap(
        clean.T,
        cmap="RdYlGn_r",
        figsize=(22, max(7, clean.shape[1] * 0.3)),
        linewidths=0,
        xticklabels=True,
        yticklabels=True,
        dendrogram_ratio=(0.1, 0.15),
        cbar_pos=(0.02, 0.8, 0.02, 0.15),
    )
    g.ax_heatmap.set_xticklabels(
        g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=5
    )
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7)
    g.figure.suptitle("Clustered performance profile (normalised ranks)", y=1.01)
    plt.show()


def plot_foundation_spread(h2h: pd.DataFrame):
    domains = h2h["domain"].dropna().unique()
    domain_pal = dict(zip(domains, plt.cm.tab10.colors[: len(domains)]))

    sub = h2h.dropna(subset=["cv"])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    for dom, grp in sub.groupby("domain"):
        ax.scatter(
            grp["other_best"],
            grp["found_median"],
            label=dom,
            alpha=0.7,
            s=35,
            color=domain_pal.get(dom, "grey"),
        )
    lims = [
        min(sub["other_best"].min(), sub["found_median"].min()),
        max(sub["other_best"].max(), sub["found_median"].max()),
    ]
    ax.plot(lims, lims, "k--", linewidth=1)
    ax.set_xlabel("Best other-model MASE")
    ax.set_ylabel("Median foundation MASE")
    ax.set_title("Below diagonal = foundation wins")
    ax.legend(fontsize=7)

    ax = axes[1]
    sc = ax.scatter(
        sub["cv"],
        sub["log_ratio"],
        alpha=0.6,
        c=sub["log_ratio"],
        cmap="RdBu_r",
        vmin=-1.5,
        vmax=1.5,
        s=35,
    )
    ax.axhline(0, color="black", linewidth=0.8)
    m, b = np.polyfit(sub["cv"], sub["log_ratio"], 1)
    xs = np.linspace(sub["cv"].min(), sub["cv"].max(), 100)
    ax.plot(xs, m * xs + b, "k--", linewidth=1.5)
    ax.set_xlabel("CV of foundation models (spread)")
    ax.set_ylabel("log(found/other)  +ve = other wins")
    plt.colorbar(sc, ax=ax, label="log_ratio")

    r, p = stats.pearsonr(sub["cv"], sub["log_ratio"])
    ax.set_title(f"Foundation spread vs advantage  (r={r:+.2f}, p={p:.3f})")

    fig.tight_layout()
    plt.show()


# ---------------------------------------------------------------------------
# Summary tables
# ---------------------------------------------------------------------------


def print_summary(h2h: pd.DataFrame):
    print("=" * 70)
    n = len(h2h)
    ow = h2h["other_wins"].sum()
    print(
        f"Best non-foundation model beats best foundation model on {ow}/{n} datasets ({100 * ow / n:.0f}%)\n"
    )

    print("--- By domain ---")
    dom = (
        h2h.groupby("domain")
        .agg(
            n=("log_ratio", "size"),
            other_wins_pct=("other_wins", lambda x: f"{100 * x.mean():.0f}%"),
            median_log_ratio=("log_ratio", "median"),
        )
        .sort_values("median_log_ratio", ascending=False)
    )
    print(dom.to_string())

    print("\n--- By frequency bucket ---")
    freq = (
        h2h.groupby("freq_bucket")
        .agg(
            n=("log_ratio", "size"),
            other_wins_pct=("other_wins", lambda x: f"{100 * x.mean():.0f}%"),
            median_log_ratio=("log_ratio", "median"),
        )
        .reindex(["sub-hourly", "hourly", "daily", "weekly+"])
    )
    print(freq.to_string())

    print("\n--- By horizon ---")
    hor = (
        h2h.groupby("ds_horizon")
        .agg(
            n=("log_ratio", "size"),
            other_wins_pct=("other_wins", lambda x: f"{100 * x.mean():.0f}%"),
            median_log_ratio=("log_ratio", "median"),
        )
        .reindex(["short", "medium", "long"])
    )
    print(hor.to_string())

    print("\n--- Datasets where non-foundation wins most (top 10) ---")
    cols = [
        "ds_name",
        "ds_freq",
        "ds_horizon",
        "domain",
        "num_variates",
        "other_best",
        "found_best",
        "log_ratio",
    ]
    print(
        h2h.sort_values("log_ratio", ascending=False)[cols]
        .head(10)
        .to_string(index=False)
    )

    print("\n--- Datasets where foundation wins most (top 10) ---")
    print(h2h.sort_values("log_ratio")[cols].head(10).to_string(index=False))

    print("\n--- Correlation: dataset features vs non-foundation advantage ---")
    for feat in ["log_freq_hours", "horizon_ord", "log_variates"]:
        sub = h2h[[feat, "log_ratio"]].dropna()
        r, p = stats.pearsonr(sub[feat], sub["log_ratio"])
        print(f"  {feat:25s}: r={r:+.3f}  p={p:.3f}")
    print("=" * 70)


## Load Raw Data


In [7]:
raw_df = load_all()

Loaded 48 models


## Build Analysis Dataframe


In [8]:
df = build_dataframe(raw_df)
df.head()


Rows: 4656, Models: 48, Datasets: 97


,dataset,model,MSE[mean],MSE[0.5],MAE[0.5],MASE[0.5],MAPE[0.5],sMAPE[0.5],MSIS,RMSE[mean],...,finetune_train_num_samples,finetune_valid_num_samples,MAAPE[0.5],ds_name,ds_freq,ds_horizon,model_name,model_type,category,org
0,bitbrains_fast_storage/5T/short,Chronos_small,1820000.0,2290000.0,171.0,0.833,2.150,0.7210,29.8,1350.0,...,NaN,NaN,NaN,bitbrains_fast_storage,5T,short,Chronos_small,pretrained,foundation,AWS AI Labs
1,bitbrains_fast_storage/H/short,Chronos_small,2640000.0,3740000.0,274.0,1.140,3.170,0.5630,29.3,1620.0,...,NaN,NaN,NaN,bitbrains_fast_storage,H,short,Chronos_small,pretrained,foundation,AWS AI Labs
2,bitbrains_rnd/5T/short,Chronos_small,1730000.0,1950000.0,132.0,1.810,1.360,0.6590,110.0,1310.0,...,NaN,NaN,NaN,bitbrains_rnd,5T,short,Chronos_small,pretrained,foundation,AWS AI Labs
3,bitbrains_rnd/H/short,Chronos_small,1980000.0,2230000.0,152.0,5.890,1.100,0.5140,199.0,1410.0,...,NaN,NaN,NaN,bitbrains_rnd,H,short,Chronos_small,pretrained,foundation,AWS AI Labs
4,bizitobs_application/10S/short,Chronos_small,6290000.0,5660000.0,955.0,3.340,0.047,0.0477,35.6,2510.0,...,NaN,NaN,NaN,bizitobs_application,10S,short,Chronos_small,pretrained,foundation,AWS AI Labs


## Head-to-Head: Statistical vs Foundation


In [9]:
h2h = build_head2head(df)
h2h.head()


,stat_best,found_best,found_median,found_std,ds_name,ds_freq,ds_horizon,domain,num_variates,log_ratio,stat_wins,winner,freq_bucket,horizon,cv,log_freq_hours,horizon_ord,log_variates
dataset,,,,,,,,,,,,,,,,,,
bitbrains_fast_storage/5T/long,1.14,0.867954,0.963292,1.067778,bitbrains_fast_storage,5T,long,Web/CloudOps,2.0,-0.272645,False,foundation,sub-hourly,long,1.108467,0.080043,2,1.098612
bitbrains_fast_storage/5T/medium,1.22,0.957508,1.056667,0.814726,bitbrains_fast_storage,5T,medium,Web/CloudOps,2.0,-0.242272,False,foundation,sub-hourly,medium,0.771033,0.080043,1,1.098612
bitbrains_fast_storage/5T/short,1.14,0.656390,0.784633,0.306152,bitbrains_fast_storage,5T,short,Web/CloudOps,2.0,-0.552029,False,foundation,sub-hourly,short,0.390185,0.080043,0,1.098612
bitbrains_fast_storage/H/short,1.32,0.930959,1.097204,1.666695,bitbrains_fast_storage,H,short,Web/CloudOps,2.0,-0.349171,False,foundation,hourly,short,1.519038,0.693147,0,1.098612
bitbrains_rnd/5T/long,3.50,3.274594,3.407760,2.108732,bitbrains_rnd,5T,long,Web/CloudOps,2.0,-0.066569,False,foundation,sub-hourly,long,0.618803,0.080043,2,1.098612


## Summary Statistics

Statistical vs. foundation model head-to-head breakdown by domain, frequency, and horizon.


In [ ]:
print_summary(h2h)


## Plot 1 – Overall Ranking

Median MASE across all datasets per model, coloured by category.


In [ ]:
plot_overall_ranking(df)


## Plot 2 – MASE by Domain

Median MASE broken down by dataset domain and model category.


In [ ]:
plot_mase_by_domain(df)


## Plot 3 – Waterfall: Statistical vs Foundation Advantage

Per-dataset log-ratio: positive (orange) means statistical wins, negative (blue) means foundation wins.


In [ ]:
plot_waterfall(h2h)


## Plot 4 – Scatter: Drivers of Statistical vs Foundation Advantage

Correlation of log-ratio with frequency, horizon, and number of variates.


In [ ]:
plot_scatter_drivers(h2h)


## Plot 5 – Boxplots: Advantage by Domain and Horizon


In [ ]:
plot_boxplots(h2h)


## Plot 6 – Who Wins by Data Frequency?


In [ ]:
plot_freq_stacked_bar(h2h)


## Plot 7 – Per-Dataset Rank Heatmap

Models sorted by mean rank; datasets sorted by domain. White lines separate domain boundaries.


In [ ]:
plot_rank_heatmap(df)


## Plot 8 – Clustered Performance Profile

Hierarchical clustering of normalised per-dataset ranks across models.


In [ ]:
plot_clustermap(df)


## Plot 9 – Foundation Spread vs Statistical Advantage

Left: scatter of best-statistical vs median-foundation MASE (below diagonal = foundation wins).  
Right: CV of foundation models vs log-ratio advantage.


In [ ]:
plot_foundation_spread(h2h)
